# Основы глубинного обучения, майнор ИАД

## Домашнее задание 1: полносвязные сети

**ФИО:**

**Факт о себе:**


## Общая информация

__Дата выдачи:__ 22.09.2025

__Мягкий дедлайн:__ 23:59MSK 12.10.2025

__Жесткий дедлайн:__ 23:59MSK 19.10.2025


## Оценивание и штрафы

Максимально допустимая оценка за работу — 10 баллов. Сдавать задание после указанного срока сдачи нельзя.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).  Если два студента сгенерировали в нейронке одинаковые либо похожие решения, это считается плагиатом и приводит к обнулению обеих работ.

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

Итогова оценка считается как

$$
min(part_1, part_2) \cdot 0.6 + max(part_1, part_2) \cdot 0.2 + part_3 \cdot 0.2
$$

где $part_1$, $part_2$ и $part_3$ - оценки за первую, вторую и третью части работы

> Также, за домашнее задание выставляется 0, если не сделано нулевое задание либо нет подробного описания ваших экспериментов в третьей части.

## Оформление

1. Обязательно фиксируйте зерно генератора случайных чисел в экспериментах. При перезапуске кода значения не должны меняться.
2. Вверху файла подпишите фамилию, имя и какой-то занимательный факт о себе.
3. Обратите внимание, что у графиков должны быть подписаны оси, заголовок графика и при необходимости обязательно наличие легенды.

> За отсутствие названий графиков и подписей к осям могут снижаться баллы. Все картинки должны быть самодостаточны и визуально удобны для восприятия, так чтобы не нужно было смотреть ваш код или знать задание, чтобы понять что на них изображено.

Из каждого проведённого эксперимента делайте выводы и фиксируйте их. Эти выводы не должны быть поверхностными и очевидными. Не будьте мудрым королём.

<br>

<center>
<img src="https://raw.githubusercontent.com/hse-ds/iad-deep-learning/refs/heads/master/2025/homeworks/king.png" width="300">
</center>

**Пример плохого вывода:** Синенькая линия идет вверх, а красная вниз. Черненькая идет вниз, а потом вверх.

<br>

<center>
<img src="https://raw.githubusercontent.com/hse-ds/iad-deep-learning/refs/heads/master/2025/homeworks/bad_lines.png" width="600">
</center>

## О задании

Вам предстоит обучить полносвязную нейронную сеть для предсказания года выпуска песни по ее аудио-признакам. Для этого мы будем использовать [Million Songs Dataset](https://samyzaf.com/ML/song_year/song_year.html).

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

import pandas as pd
import numpy as np
import random

from tqdm.notebook import tqdm
from IPython.display import clear_output
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

Начнем с того, что скачаем и загрузим данные:

In [5]:
!wget -O yearpredictionmsd.zip https://archive.ics.uci.edu/static/public/203/yearpredictionmsd.zip

--2025-10-19 12:51:01--  https://archive.ics.uci.edu/static/public/203/yearpredictionmsd.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘yearpredictionmsd.zip’

yearpredictionmsd.z     [           <=>      ]  24.60M   287KB/s               ^C


In [6]:
df = pd.read_csv('yearpredictionmsd.zip', header=None)
df.head()

BadZipFile: File is not a zip file

Посмотрим на статистики по данным.

In [ ]:
df.describe()

Целевая переменная, год выпуска песни, записана в первом столбце. Посмотрим на ее распределение.

In [ ]:
plt.hist(df.iloc[:, 0], bins=20)
plt.xlabel('year')
plt.ylabel('count')
plt.show()
print(f'Range: {df.iloc[:, 0].min()} - {df.iloc[:, 0].max()}')
print(f'Unique values: {np.unique(df.iloc[:, 0]).size}')

Разобьем данные на обучение и тест (не меняйте здесь ничего, чтобы сплит был одинаковым у всех).

In [ ]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

train_size = int(0.75 * X.shape[0])

X_train = X[:train_size, :]
y_train = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]

X_train.shape, X_test.shape

## Полезные советы:

- Если вы сразу реализуете обучение на GPU, то у вас будет больше времени на эксперименты, так как любые вычисления будут работать быстрее. Google Colab предоставляет несколько GPU-часов (обычно около 8-10) в сутки бесплатно.

- Если вы чего-то не знаете, не стесняйтесь гуглить. В интернете очень много полезной информации, туториалов и советов по глубинному обучению и `pytorch`. Но не забывайте, что за списанный код без ссылки на источник последует наказание.

- Чтобы отладить код, можете обучаться на небольшой части данных или даже на одном батче. Если лосс на обучающей выборке не падает, то что-то точно идет не так.

- Пользуйтесь утилитами, которые вам предоставляет `pytorch` (например, `Dataset` и `Dataloader`). Их специально разработали для упрощения разработки пайплайна обучения.

- Скорее всего, вы захотите отслеживать прогресс обучения. Для создания прогресс-баров есть удобная библиотека `tqdm`.

- Быть может, вы захотите, чтобы графики рисовались прямо во время обучения. Можете воспользоваться функцией [clear_output](http://ipython.org/ipython-doc/dev/api/generated/IPython.display.html#IPython.display.clear_output), чтобы удалять старый график и рисовать новый на его месте.

- При желании вы можете логгировать метрики обучения и свои эксперименты в WandB либо любой другой сервис. Не забудьте приложить к тетрадке ссылку на результаты экспериментов либо скришноты графиков с пояснениями, что проверяющий должен на них увидеть.

- Финальное значение тестовой метрики для удобства проверки выведите в тетрадке.

## Задание 0 (0 баллов, но при невыполнении максимальная оценка за всю работу &mdash; 0 баллов)

Мы будем использовать RMSE как метрику качества. Прежде чем обучать нейронные сети, нам нужно проверить несколько простых бейзлайнов, чтобы было с чем сравнить более сложные алгоритмы. Для этого бучите `Ridge` регрессию из `sklearn`. Кроме того, посчитайте качество при наилучшем константном прогнозе.

Для выполнения данного задания (и всех последующих) предобработайте данные.

1. Зафиксируйте random_seed везде где только возможно. Вам предоставлена функция для этого, однако вы можете дополнить ее своими дополнениями.
2. Обучите `StandertScaler` и предобработайте ваши данные. В следующих заданиях можете использовать другой `scaler` или вообще отказаться от него.


In [ ]:
def set_global_seed(seed: int) -> None:
    """Set global seed for reproducibility.
    :param int seed: Seed to be set
    """
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # также можно зафиксировать seed для Dataloader
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# Сид для каждого worker в Dataloader
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = set_global_seed(42)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
scaler = StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
model = Ridge(random_state=42)
model.fit(X_train, y_train)
rmse_for_model = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
print(rmse_for_model)

# your code here  ⟅⎰᨟﹏᨟⎱⟆

Лучшая константа для RMSE это среднее. Используйте среднее, расчитанное на трэйне в качестве прогноза для теста и посчитайте для такой наивной модели RMSE.

In [ ]:
 # your code here  ⟅⎛ꌩωꌩ⎞⟆
constant= np.full_like(y_test, fill_value=np.mean(y_train))
best_rmse = np.sqrt(mean_squared_error(y_test, constant))
print(best_rmse)

Теперь приступим к экспериментам с нейросетями. Для начала отделим от данных валидацию. Тестовую выборку мы будем использовать только для того, чтобы измерить итоговую метрику качества модели.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=0xE2E4)
X_train.shape, X_val.shape

## Часть I. Обучаем линейную регрессию (максимум 10 баллов)

**Задание 1 (10 баллов):** Обучите в `pytorch` линейную регрессию.

- Создайте модель линейной регрессии, которая будет состоять только из одного `Linear()` слоя.
   
- Напишите цикл обучения вашей линейной регрессии. В нем реализуйте подсчет функции потерь, сделайте шаг градиентного спуска. Запрещено использовать готовые оптимизаторы и loss-функции из библиотеки `pytorch`. Для подсчета градиента воспользуйтесь методом backward.
   
- Запустите обучение на 10 эпохах, после каждой проверяйте значение целевой метрики на тестовой выборке.
   
- Выведите на экран графики метрики и значения функции потерь на тестовой и обучающей выборке.

В данном задании нет цели побить какой-то порог по метрике. Ваша задача &mdash; убедиться в том, что ваш рукописный цикл обучения работает. Для ускорения вычислений и обучения модели можете брать только срез данных, а не весь датасет.

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

In [ ]:
class LinearRegression(nn.Module):
    def __init__(self):
        super(LinearRegression,self).__init__()
        self.linear = nn.Linear(90, 1)

    def forward(self, x):
        return self.linear(x)
model = LinearRegression()
def mse(y_pred, y):
    return ((y_pred - y)**2).mean()
def rmse(y_pred, y):
    return torch.sqrt(((y_pred - y) ** 2).mean())

train_losses, val_losses,train_rmses,val_rmses = [],[],[],[]
# я специально поставила очень маленькую learning rate так как при больших значениях уже на четвертой эпохе ошибка на обеих выборках становилась бесконечостью
lr = 0.0000001
for epoch in range(1,11):
    model.train()
    train_pred = model(X_train_tensor)
    loss = mse(train_pred, y_train_tensor)
    train_losses.append(loss.item())
    train_rmse=rmse(train_pred,y_train_tensor)
    train_rmses.append(train_rmse.item())
    model.zero_grad()
    loss.backward()
    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_tensor)
        val_loss = mse(val_pred, y_val_tensor)
        val_rmse=rmse(val_pred,y_val_tensor)
    val_losses.append(val_loss.item())
    val_rmses.append(val_rmse.item())
    print(f'epoch: {epoch}, train loss: {loss.item():.4f}, val loss: {val_loss.item():.4f}, RMSE: {val_rmse.item():.4f}')



In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(range(1, 11), train_losses, label='train')
plt.plot(range(1, 11), val_losses, label='validation')
plt.xlabel('epochs')
plt.ylabel('loss (mse)')
plt.title('loss on train and validation')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, 11), train_rmses, label='train')
plt.plot(range(1, 11), val_rmses, label='Validation')
plt.xlabel('epochs')
plt.ylabel('rmse')
plt.title('rmse on train and validation')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


Видим, что lossы огромные -- скорее всего это из-за неотнормализованной целевой переменной. Из-за этого пришлось очень сильно уменьшить learning rate. Также видим, что rmse очень большие (больше 1998), но модель все равно обучается -- на каждой эпохе rmse чуть-чуть уменьшается. (в этом задании много чего подсмотрела во втором семинарском ноутбуке)

## Часть II. Заводим нейронную сеть (максимум 10 баллов)

Ниже нам предстоит реализовать довольно много различных нейросетей и поставить целую серию экспериментов. Чтобы это всё происходило без боли и страданий, нам нужно держать код в удобном виде.

При решении заданий вы можете придерживаться любой адекватной струкуры кода, но мы советуем воспользоваться сигнатурами функций, которые приведены ниже. При необходимости вы можете добавить в них любые нужные вам аргументы и любой нужный функционал. Более того, хорошей практикой является не делать эти функции слишком громоздкими и выносить разные хитрые штуки в отдельные функции.

Здесь функции написала аналогичные тем, что во 2 семинарском ноутбуке (+немного изменила функцию predict, возвращаю там еще y_true):

In [ ]:

from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [ ]:
def training_epoch(model, optimizer, criterion, train_loader):
    """Одна эпоха обучения
    params:
        model - torch.nn.Module to be fitted
        optimizer - model optimizer
        criterion - loss function from torch.nn
        train_loader - torch.utils.data.Dataloader with train set
    """
    model.train()
    train_loss= 0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    return train_loss
    raise NotImplementedError


@torch.no_grad()

def validation_epoch(model, criterion, val_loader):
    """Одна эпоха валидации модели
    params:
        model - torch.nn.Module to be fitted
        criterion - loss function from torch.nn
        val_loader - torch.utils.data.Dataloader with test set
                      (if you wish to validate during training)
    """
    model.eval()
    val_loss = 0
    for inputs, targets in val_loader:
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        val_loss += loss.item()

    val_loss /= len(val_loader)
    return val_loss
    # your code here   ฅ^•ﻌ•^ฅ

    raise NotImplementedError


@torch.no_grad()

def predict(model, data_loader):
    """ Предсказания модели
    params:
        model - torch.nn.Module to be evaluated on test set
        criterion - loss function from torch.nn
        data_loader - torch.utils.data.Dataloader with test set
    ----------
    returns:
        predicts - torch.tensor with shape (len(test_loader.dataset), ),
                   which contains predictions for test objects
    """

    # your code here  =^･ω･^=
    model.eval()
    predicts = []
    y_true=[]

    with torch.no_grad():
        for inputs, labels in data_loader:
            predicts.append(model(inputs).cpu())
            y_true.append(labels.cpu())
    return torch.cat(predicts, dim=0), torch.cat(y_true, dim=0)


def train(model, optimizer, criterion, train_loader, val_loader, epochs):
    """ Обучение модели
    params:
        model - torch.nn.Module to be fitted
        optimizer - model optimizer
        criterion - loss function from torch.nn
        train_loader - torch.utils.data.Dataloader with train set
        val_loader - torch.utils.data.Dataloader with test set
                      (if you wish to validate during training)
        epochs - number of training epochs
    """

    # your code here  ¯\_(ツ)_/¯
    for epoch in range(1,epochs+1):
        train_loss = training_epoch(model, optimizer, criterion, train_loader)
        val_loss = validation_epoch(model, criterion, val_loader)

        print(f'Epoch {epoch}, loss on train: {train_loss:.4f}, loss on validation: {val_loss:.4f}')



In [ ]:
def rmse(predictions, targets):
    return np.sqrt(np.mean((predictions - targets) ** 2))

**Задание 2 (2 балла)**

Попробуем обучить нашу первую нейронную сеть. Здесь целевая переменная дискретная &mdash; это год выпуска песни. Поэтому будем учить сеть на классификацию.

- В качестве архитектуры сети возьмите два линейных слоя с активацией ReLU между ними c числом скрытых нейронов, равным 128.
- Используйте SGD с `lr=1e-3`.
- Возьмите размер мини-батча около 32-64, примерно 3-4 эпох обучения должно быть достаточно.
- Также преобразуйте целевую переменную так, чтобы ее значения принимали значения от $0$ до $C-1$, где $C$ &mdash; число классов (лучше передайте преобразованное значение в DataLoader, исходное нам еще пригодится)
- В качестве метрики качества мы используем RMSE. При его подсчёте вам нужно заменить предсказанный нейросеткой класс на конкретный год выпуска песни и использовать его как прогноз.

In [ ]:

g=set_global_seed(42)
class ClassificationNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ClassificationNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes))

    def forward(self, x):
        return self.model(x)

year_indexes_train = {year: i for i, year in enumerate(np.unique(y_train))}
indexes_train = np.array([year_indexes_train[year] for year in y_train])

year_indexes_val = {year: i for i, year in enumerate(np.unique(y_val))}
indexes_val = np.array([year_indexes_val[year] for year in y_val])

train_dataset = TensorDataset(torch.tensor(X_train,dtype=torch.float32), torch.tensor(indexes_train).long())
val_dataset = TensorDataset(torch.tensor(X_val,dtype=torch.float32), torch.tensor(indexes_val).long())
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, generator=g, worker_init_fn=seed_worker)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32,generator=g, worker_init_fn=seed_worker)

model = ClassificationNN(X_train.shape[1], len(np.unique(indexes_train)))
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)

train(model, optimizer, criterion, train_loader, val_loader, epochs=4)
predictions, y_true = predict(model, val_loader)
predicted_classes = torch.argmax(predictions, dim=1)
index_to_year = {y: x for x, y in year_indexes_train.items()}
predicted_years = [index_to_year[i] for i in predicted_classes.cpu().numpy() if i in index_to_year]
true_years=[index_to_year[i.item()] for i in y_true]
rmse_classification= rmse(np.array(predicted_years), np.array(true_years))
print(f'RMSE of classification on validation: {rmse_classification:.2f}')




**Задание 3 (1 балл).** Прокомментируйте ваши наблюдения. Удалось ли побить бейзлайн? Как вы думаете, хорошая ли идея учить классификатор для этой задачи? Почему?

**Ответ:** Видно, что на каждой эпохе значения СrossEntropyLoss уменьшается, как на трэйне, так и на таргете, то есть модель все лучше различает классы. Однако RMSE у константной модели оказалось значительно лучше, чем у классификатора. Мне кажется, в контексте данной задачи не стоит выбирать классификацию:
для классификации нет большой разницы между классами (ошибка между 2004 и 2003 такая же, как между 2004 и 1990), хотя изначально нам важен порядок годов. Интуитивно музыка 90-х сильно отличается от музыки 2000-x и 2010-x, в классификации же все классы равноудалены друг от друга и модель одинаково наказывает, что за ошибку в 20 лет, что за 1 год.


**Задание 4 (2 балла).** Теперь попробуем решать задачу как регрессию. Обучите нейронную сеть на MSE.

- Используйте такие же гиперпараметры обучения.
- Когда передаете целевую переменную в DataLoader, сделайте reshape в (-1, 1).
- Если что-то пойдет не так, можете попробовать меньшие значения `lr`.

In [ ]:
y_train = y_train.reshape(-1, 1)
y_val = y_val.reshape(-1, 1)

g=set_global_seed(42)
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,generator=g, worker_init_fn=seed_worker)
val_loader = DataLoader(val_dataset, batch_size=32,generator=g, worker_init_fn=seed_worker)

class RegressionNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.model(x)
model = RegressionNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=1e-5)
train(model, optimizer, criterion, train_loader, val_loader, epochs=3)
predictions,y_true = predict(model, val_loader)
rmse_regression = rmse(predictions.numpy().flatten(),y_true.numpy().flatten())
print(f'RMSE of regression: {rmse_regression:.2f}')

**Задание 5 (1 балл).** Получилось ли у вас стабилизировать обучение? Помогли ли меньшие значения `lr`? Стало ли лучше от замены классификации на регрессию? Как вы думаете, почему так происходит? В качестве подсказки можете посмотреть на распределение целевой переменной и магнитуду значений признаков.

**Ответ:** Уменьшение lr сильно помогло, так как при больших значениях был взрыв градиента. Здесь видим, что на первой эпохе было огромное значение loss, но далее все более менее стабилизировалось, хотя lossы все равно большие. Основной проблемой этой модели является то, что данные X_train/val уже нормализованы и находятся между 0 и 1, а целевая переменная (годы) неотнормализована и больше 1500, из-за чего у нас большие градиенты и даже с очень маленьким lr=10^-5 градиентный спуск делает очень большие шаги. RMSE значительно уменьшилась по сравнению с регрессией и сейчас этот показатель близок к бейзлайну. Регрессия явно подходит больше, чем классификация по описанным выше причинам (регрессия чувствует разницу между 1 год и 20 лет).

**Задание 6 (1 балл).** Начнем с того, что попробуем отнормировать целевую переменную. Для этого воспользуемся min-max нормализацией, чтобы целевая переменная принимала значения от 0 до 1. Реализуйте функции `normalize` и `denormalize`, которые, соответственно, нормируют целевую переменную и применяют обратное преобразование. Минимум и максимум оцените по обучающей выборке (то есть эти константы должны быть фиксированными и не зависеть от передаваемой выборки).

In [ ]:
min_value = np.min(y_train)
max_value = np.max(y_train)
def normalize(sample):
    """
    Min-max normalization to convert sample to [0, 1] range
    """
    if min_value!=max_value:
      return (sample - min_value) / (max_value - min_value)
    else:
      return np.zeros_like(sample)

def denormalize(sample):
    """
    Denormalize sample from [0, 1] to initial range
    """
    return sample * (max_value - min_value) + min_value
    # your code here ( ⚆ ω ⚆)

Здесь я не очень поняла, что делать в случае, когда в трэйне все значения одинаковые -- поэтому вернула нули. Но на практике это очень редко и этого нет в этой домашке, так что в данном случае функции действительно обратны друг другу.

**Задание 7 (1 балл)** Теперь повторите эксперимент из **задания 4**, обучаясь на нормированной целевой переменной. Сделаем также еще одно изменение: добавим сигмоидную активацию после последнего линейного слоя сети. Таким образом мы гарантируем, что нейронная сеть предсказывает числа из промежутка $[0, 1]$. Использование активации - довольно распространенный прием, когда мы хотим получить числа из определенного диапазона значений.

In [ ]:
y_train_normalized = normalize(y_train).reshape(-1, 1)
y_val_normalized = normalize(y_val).reshape(-1, 1)
g=set_global_seed(42)
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train_normalized, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val_normalized, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,generator=g, worker_init_fn=seed_worker)
val_loader = DataLoader(val_dataset, batch_size=32,generator=g, worker_init_fn=seed_worker)

class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model = RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)
train(model, optimizer, criterion, train_loader, val_loader, epochs=3)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid = rmse(predictions, y_true)
print(f'RMSE of regression: {rmse_sigmoid:.2f}')
# your code here  ლ(ಠ益ಠლ)

**Задание 8 (2 балла).** На этот раз попробуем отнормировать не только целевую переменную, но и сами данные, которые подаются сети на вход. Для них будем использовать нормализацию через среднее и стандартное отклонение. Преобразуйте данные и повторите прошлый эксперимент. Скорее всего, имеет смысл увеличить число эпох обучения.

In [ ]:
X_train_normalized = scaler.transform(X_train)
X_val_normalized = scaler.transform(X_val)
g=set_global_seed(42)
train_dataset = TensorDataset(torch.tensor(X_train_normalized, dtype=torch.float32), torch.tensor(y_train_normalized, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val_normalized, dtype=torch.float32), torch.tensor(y_val_normalized, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,generator=g, worker_init_fn=seed_worker)
val_loader = DataLoader(val_dataset, batch_size=32, generator=g, worker_init_fn=seed_worker)

model = RegressionSigmoidNN(X_train_normalized.shape[1])
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)
train(model, optimizer, criterion, train_loader, val_loader, epochs=9)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'Validation RMSE: {rmse_sigmoid_normalized:.2f}')
 # your code here  ( ͡° ͜ʖ ͡°)

Мы уже до этого обучались на отнормализованных данных, поэтому здесь я это просто еще раз прописала (хотя по сути это ненужно, так как они уже отнормализованы). В любом случае видим, что сигмоидная активация+большее количество эпох сильно улучшили rmse на валидации.

Если вы все сделали правильно, то у вас должно было получиться качество, сравнимое с `Ridge` регрессией.

**Мораль:** как видите, нам пришлось сделать очень много хитрых телодвижений, чтобы нейронная сеть работала хотя бы так же, как и простая линейная модель. Здесь, конечно, показан совсем экстремальный случай, когда без нормализации данных нейронная сеть просто не учится. Как правило, в реальности завести нейронную сеть из коробки не очень сложно, но вот заставить ее работать на полную &mdash; куда более трудоемкая задача. Написание пайплайнов обучения нейросетевых моделей требует большой аккуратности, а дебаг часто превращается в угадайку. К счастью, очень часто на помощь приходит интуиция, и мы надеемся, что вы сможете выработать ее в течение нашего курса. Начнем с двух советов, которые стоит принять на вооружение:

- Обязательно начинаем любые эксперименты с бейзлайнов: без них мы бы не поняли, что нейронная сеть не учится в принципе.
- При постановке эксперментов старайтесь делать минимальное количество изменений за раз (в идеале одно!): только так можно понять, какие конкретно изменения влияют на результат.

## Часть III. Улучшаем нейронную сеть (максимум 10 баллов)

Продолжим экспериментировать с нейронной сетью, чтобы добиться еще лучшего качества.

**Задание 9 (1 балл).** Давайте попробуем другие оптимизаторы. Обучите нейросеть с помощью SGD+momentum и Adam. Опишите свои наблюдения и в дальнейших запусках используйте лучший оптимизатор. Для Adam обычно берут learning rate поменьше, в районе $10^{-3}$.

In [ ]:
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
train(model, optimizer, criterion, train_loader, val_loader, epochs=7)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'RMSE: {rmse_sigmoid_normalized:.2f}')

In [ ]:
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train(model, optimizer, criterion, train_loader, val_loader, epochs=7)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'RMSE: {rmse_sigmoid_normalized:.2f}')

Оба оптимизатора очень хорошо уменьшили rmse (больше чем на 1) и lossы у обоих очень маленькие (у Adama даже чуть меньше). По RMSE Adam оказался лучше, поэтому далее будем использовать его.

**Задание 10 (1 балл).** Теперь сделаем нашу нейронную сеть более сложной. Попробуйте сделать сеть:

- более широкой (то есть увеличить размерность скрытого слоя, например, вдвое)
- более глубокой (то есть добавить еще один скрытый слой)

In [ ]:
class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train(model, optimizer, criterion, train_loader, val_loader, epochs=7)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'RMSE: {rmse_sigmoid_normalized:.2f}')

Опишите, как увеличение числа параметров модели влияет на качество на обучающей и валидационной выборках (без их описания за работу ставится ноль баллов)

__Ваше подробное описание:__ Мы увеличили размерность скрытого слоя вдвое и добавили еще один слой, далее запустили обучения на 7 эпохах. Переобучения это не вызвало -- можно заметить, что loss на трэйне стабильно уменьшается, как и loss на валидации (если бы было переобучение, то на валидации были бы скачки вверх). По сравнению с более простой моделью (на один слой меньше + менее широкая) rmse стал меньше почти на 0.1, поэтому в данном случае усложнение модели положительно сказалось на предсказаниях.

**Задание 11 (1 балл).** Как вы должны были заметить, более сложная модель стала сильнее переобучаться. Попробуем разные методы регуляризации, чтобы бороться с переобучением. Проведите два эксперимента:

- Добавьте слой дропаута с параметром $p=0.2$ после каждого линейного слоя, кроме последнего.
- Попробуйте batch-нормализацию вместо дропаута. Строго говоря, batch-нормализация не является методом регуляризации, но никто не запрещает нам экспериментировать с ней.

In [ ]:
class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size, dropout_rate=0.2):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train(model, optimizer, criterion, train_loader, val_loader, epochs=7)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'Validation RMSE: {rmse_sigmoid_normalized:.2f}')

In [ ]:
class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
train(model, optimizer, criterion, train_loader, val_loader, epochs=7)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'Validation RMSE: {rmse_sigmoid_normalized:.2f}')

Опишите результаты экспериментов (без их описания за работу ставится ноль баллов)

__Ваше подробное описание:__ И dropout, и batch нормализация уменьшили rmse, но вторая уменьшила rmse сильнее, поэтому далее выберем ее. Также заметим, что пока нет особых признаков переобучения -- на каждой эпохе loss на валидации, что у dropout что у batch-нормализации стабильно падает.

**Задание 12 (1 балл).** Теперь, когда мы определились с выбором архитектуры нейронной сети, пора заняться рутиной DL-инженера &mdash; перебором гиперпараметров. Подберите оптимальное значение lr по значению RMSE на валидации (по логарифмической сетке, достаточно посмотреть 3-4 значения). Затем подберите оптимальное значение weight decay для данного lr (тоже по логарифмической сетке, типичные значения этого параметра лежат в диапазоне $[10^{-6}, 10^{-3}]$, но не забудьте включить нулевое значение в сетку). Постройте графики зависимости RMSE на трейне и на валидации от значений параметров. Прокомментируйте получившиеся зависимости.

In [ ]:
rates = [1e-5, 1e-4, 1e-3]
lr_rmse = []

for lr in rates:
    model = RegressionSigmoidNN(X_train_normalized.shape[1])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train(model, optimizer, criterion, train_loader, val_loader, epochs=4)

    predictions,y_true= predict(model, train_loader)
    predictions=denormalize(predictions).numpy().flatten()
    y_true=denormalize(y_true).numpy().flatten()
    rmse_train = rmse(predictions, y_true)

    predictions,y_true= predict(model, val_loader)
    predictions=denormalize(predictions).numpy().flatten()
    y_true=denormalize(y_true).numpy().flatten()
    rmse_val = rmse(predictions, y_true)
    lr_rmse.append({
        'lr': lr,
        'rmse_train': rmse_train,
        'rmse_val': rmse_val
    })
best_lr = min(lr_rmse, key=lambda x: x['rmse_val'])['lr']
print(f"best_lr: {best_lr}")

plt.figure(figsize=(12,6))
plt.subplot(1, 2, 1)
plt.semilogx([r['lr'] for r in lr_rmse], [r['rmse_train'] for r in lr_rmse], 'o-b', label='train')
plt.semilogx([r['lr'] for r in lr_rmse], [r['rmse_val'] for r in lr_rmse], 'o-r', label='validation')
plt.xlabel('learning rate')
plt.ylabel('rmse')
plt.title('dependence of rmse on learning rate')
plt.legend()
plt.grid(True)


In [ ]:
wd_values = [0, 1e-6, 1e-4, 1e-3]
wd_rmse = []

for wd in wd_values:
    model = RegressionSigmoidNN(X_train_normalized.shape[1])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=best_lr, weight_decay=wd)
    train(model, optimizer, criterion, train_loader, val_loader, epochs=3)

    predictions,y_true= predict(model, train_loader)
    predictions=denormalize(predictions).numpy().flatten()
    y_true=denormalize(y_true).numpy().flatten()
    rmse_train = rmse(predictions, y_true)

    predictions,y_true= predict(model, val_loader)
    predictions=denormalize(predictions).numpy().flatten()
    y_true=denormalize(y_true).numpy().flatten()
    rmse_val = rmse(predictions, y_true)

    wd_rmse.append({
        'wd': wd,
        'rmse_train': rmse_train,
        'rmse_val': rmse_val})

best_wd = min(wd_rmse, key=lambda x: x['rmse_val'])['wd']
print(f"best weight decay: {best_wd} ")



In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 2)
plt.semilogx([r['wd'] for r in wd_rmse], [r['rmse_train'] for r in wd_rmse], 'o-b', label='train')
plt.semilogx([r['wd'] for r in wd_rmse], [r['rmse_val'] for r in wd_rmse], 'o-r', label='validation')
plt.xlabel('weight decay')
plt.ylabel('rmse')
plt.title('dependence of rmse on weight decay')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()


Опишите результаты экспериментов (без их описания за работу ставится ноль баллов)

__Ваше подробное описание:__  Мы перебрали три значения для lr и 4 значения для weight decay, обучаясь на 3-4 эпохах (такое маленькое количество эпох, так как иначе слишком долго). По результатам наших экспериментов опьтмальное значение lr -- это 10^-3, weight decay -- 0 (rmse на валидации будет меньше 8.9 и 8.8 соответственно). Также по построенным графикам можно заметить, что чем меньше learning rate, тем больше rmse. В случае же с weight decay наоборот (чем он меньше, тем ниже rmse).

> Как вы могли заметить, еще одна рутина DL-инженера &mdash; утомительное ожидание обучения моделей.




**Задание 13 (6 баллов).**

Думаю направление размышлений вы поняли. Постарайтесь с помощью своих экспериментов выбить максимально возможное значение RMSE на тестовой выборке. Соотношение между полученным значением метрики на тестовой выборке и баллами за задание следующее:

- $\text{RMSE} \le 8.90 $ &mdash; 2 балла
- $\text{RMSE} \le 8.80 $ &mdash; 4 балла
- $\text{RMSE} \le 8.75 $ &mdash; 6 баллов

**Различные трюки, которые можно попробовать:**

1. Попробуйте делать во время обучения раннюю остановку обучения и сохранять модель в тот момент, когда качество на валидации начало ухудшаься, то есть модель начала переобучаться
2. Попробуйте усложнить архитектуру нейросет
    - Больше/меньше нейронов
    - Больше/меньше слоёв
    - Другие функции активации (tanh, relu, leaky relu, elu etc)
    - Регуляризация (dropout, l1,l2)
3. Попробуйте другие оптимизаторы, а также смену скорости обучения по расписанию.

И это далеко не полный список. Обратите внимание, что делать grid_search для больших сеток это довольно времязатратное занятие... Попробовать несколько значений, как мы делали в заданиях выше, адекватно, но делать какой-то огромный перебор будет самоубийством.

Логгируйте свои эксперименты. За один прогон пробуйте одно изменение. Иначе будет непонятно какие именно изменения улучшили качество, а какие ухудшили.

In [ ]:
y_test_normalized = normalize(y_test).reshape(-1, 1)
X_test_normalized = scaler.transform(X_test)
test_dataset = TensorDataset(torch.tensor(X_test_normalized, dtype=torch.float32), torch.tensor(y_test_normalized, dtype=torch.float32))
test_loader = DataLoader(test_dataset, batch_size=32,generator=g, worker_init_fn=seed_worker)


Посмотрим, какое rmse выдает наша модель на тесте:

In [ ]:
model = RegressionSigmoidNN(X_train_normalized.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=best_lr,weight_decay=best_wd)
train(model, optimizer, criterion, train_loader, val_loader, epochs=6)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_val= rmse(predictions, y_true)
print('Validation: ',rmse_val)
predictions,y_true= predict(model, test_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_test = rmse(predictions, y_true)
print('Test: ', rmse_test)

Уже хорошо, но попробуем провести еще несколько экспериментов, чтобы понять, что улучшит, а что ухудшит ее. Добавим dropout_rate=0.3:

In [ ]:
class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size, dropout_rate=0.3):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
train(model, optimizer, criterion, train_loader, val_loader, epochs=6)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'Validation: {rmse_sigmoid_normalized:.2f}')

Сделаем ее еще шире:

In [ ]:
class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
train(model, optimizer, criterion, train_loader, val_loader, epochs=6)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'Validation: {rmse_sigmoid_normalized:.2f}')

Сделаем ее глубже:

In [ ]:
class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
train(model, optimizer, criterion, train_loader, val_loader, epochs=6)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'Validation RMSE: {rmse_sigmoid_normalized:.2f}')

In [ ]:
class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
train(model, optimizer, criterion, train_loader, val_loader, epochs=6)
predictions,y_true= predict(model, val_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'Validation: {rmse_sigmoid_normalized:.2f}')

In [ ]:
class RegressionSigmoidNN(nn.Module):
    def __init__(self, input_size):
        super(RegressionSigmoidNN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
model= RegressionSigmoidNN(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
train(model, optimizer, criterion, train_loader, val_loader, epochs=6)
predictions,y_true= predict(model, test_loader)
predictions=denormalize(predictions).numpy().flatten()
y_true=denormalize(y_true).numpy().flatten()
rmse_sigmoid_normalized = rmse(predictions, y_true)
print(f'Validation: {rmse_sigmoid_normalized:.2f}')

Опишите результаты экспериментов (без их описания за работу ставится ноль баллов)

__Ваше подробное описание:__

## Бонус (0.1 балла)

Прикрепите фотографию того, как вы начали этот сентябрь. Какую самую классную эмоцию вы испытали за прошедший месяц?

__место для картики и эмоции__
